In [1]:
import re
import pandas as pd
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import classification_report

ds = load_dataset("zeroshot/twitter-financial-news-sentiment")
val = ds["validation"].to_pandas()
val["label_name"] = val["label"].map({0: "bearish", 1: "bullish", 2: "neutral"})

clf = pipeline("text-classification", model="ProsusAI/finbert", top_k=None)

def clean(t):
    t = re.sub(r"https?://\S+", "", t)
    return t.replace("…", "").strip()

sample = val.sample(300, random_state=1).copy()
outs = clf(sample["text"].map(clean).tolist(), batch_size=16, truncation=True)

def to_dict(o):
    return {x["label"]: x["score"] for x in o}

sample["score"] = [to_dict(o)["positive"] - to_dict(o)["negative"] for o in outs]
sample["pred_label"] = [
    {"positive": "bullish", "negative": "bearish", "neutral": "neutral"}[max(o, key=lambda x: x["score"])["label"]]
    for o in outs
]

print(classification_report(sample["label_name"], sample["pred_label"]))
print(pd.crosstab(sample["label_name"], sample["pred_label"]))

README.md:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

sent_train.csv:   0%|          | 0.00/859k [00:00<?, ?B/s]

sent_valid.csv:   0%|          | 0.00/217k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

              precision    recall  f1-score   support

     bearish       0.51      0.74      0.60        42
     bullish       0.52      0.48      0.50        56
     neutral       0.82      0.76      0.79       202

    accuracy                           0.70       300
   macro avg       0.62      0.66      0.63       300
weighted avg       0.72      0.70      0.71       300

pred_label  bearish  bullish  neutral
label_name                           
bearish          31        0       11
bullish           6       27       23
neutral          24       25      153
